In [1]:
import psycopg2
import subprocess

<frozen importlib._bootstrap>:491: RuntimeWarning: The global interpreter lock (GIL) has been enabled to load module 'psycopg2._psycopg', which has not declared that it can run safely without the GIL. To override this behavior and keep the GIL disabled (at your own risk), run with PYTHON_GIL=0 or -Xgil=0.


In [3]:
# Add family size information to the DataFrame, pull this from the database
DB = {
    "host":     "petadex.ccz9y6yshbls.us-east-1.rds.amazonaws.com",
    "port":     5432,
    "database": "petadex",
    "user":     "readonly_user",
    "password": "petadex",
}

conn = psycopg2.connect(**DB)
cur  = conn.cursor()

In [4]:
cur.execute("""SELECT DISTINCT family, COUNT(*)
            FROM enzyme_taxonomy
            WHERE family IS NOT NULL
            GROUP BY family
            """)
families = cur.fetchall()

In [5]:
OUTPUT_DIR = "./data/families"

In [6]:
import os

family_ids = [f[0] for f in families]
batch_size = 1000

for i in range(0, len(family_ids), batch_size):
    batch = family_ids[i:i+batch_size]
    
    # 1. Fetch the batch of families and their corresponding enzyme sequences
    cur.execute("""
        SELECT family, enzyme_fastaa.enzyme_id, translated_sequence 
        FROM enzyme_taxonomy
        JOIN enzyme_fastaa ON enzyme_taxonomy.enzyme_id = enzyme_fastaa.enzyme_id
        WHERE family IN %s
    """, (tuple(batch),))
    results = cur.fetchall()

    # 2. Group the results by family
    family_data = {}
    for fam_id, enz_id, seq in results:
        if fam_id not in family_data:
            family_data[fam_id] = []
        family_data[fam_id].append(f">{enz_id}\n{seq}\n")

    # 3. Write each family to its own .fasta file
    for fam_id, fasta_lines in family_data.items():
        fasta_path = os.path.join(OUTPUT_DIR, f"{fam_id}.fasta")
        with open(fasta_path, "w") as f:
            f.writelines(fasta_lines)

    print(f"Processed batch {i//batch_size + 1}: Created {len(family_data)} FASTA files.")

Processed batch 1: Created 1000 FASTA files.
Processed batch 2: Created 1000 FASTA files.
Processed batch 3: Created 1000 FASTA files.
Processed batch 4: Created 1000 FASTA files.
Processed batch 5: Created 1000 FASTA files.
Processed batch 6: Created 1000 FASTA files.
Processed batch 7: Created 1000 FASTA files.
Processed batch 8: Created 1000 FASTA files.
Processed batch 9: Created 1000 FASTA files.
Processed batch 10: Created 1000 FASTA files.
Processed batch 11: Created 1000 FASTA files.
Processed batch 12: Created 1000 FASTA files.
Processed batch 13: Created 1000 FASTA files.
Processed batch 14: Created 1000 FASTA files.
Processed batch 15: Created 1000 FASTA files.
Processed batch 16: Created 1000 FASTA files.
Processed batch 17: Created 1000 FASTA files.
Processed batch 18: Created 1000 FASTA files.
Processed batch 19: Created 1000 FASTA files.
Processed batch 20: Created 1000 FASTA files.
Processed batch 21: Created 1000 FASTA files.
Processed batch 22: Created 1000 FASTA file

In [ ]:
for family in families:
    family_id = family[0]
    cur.execute("""SELECT enzyme_fastaa.enzyme_id, translated_sequence
                FROM enzyme_taxonomy
                JOIN enzyme_fastaa ON enzyme_taxonomy.enzyme_id = enzyme_fastaa.enzyme_id
                WHERE family = %s
                """, (family_id,))
    rows = cur.fetchall()

    # Write the sequences to FASTA
    fasta_path = f"{OUTPUT_DIR}/{family_id}.fasta"
    with open(fasta_path, "w") as f:
        for seq_id, seq in rows:
            f.write(f">{seq_id}\n{seq}\n")

    # Run MUSCLE to align the sequences
    input_file = fasta_path
    output_file = f"{OUTPUT_DIR}/{family_id}_aligned.fasta"
    cmd = ["muscle", "-align", input_file, "-output", output_file]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode == 0:
        print("Alignment successful!")
    else:
        print(f"Error: {result.stderr}")
    

NameError: name 'toytree' is not defined

In [4]:
input_file = "input.fasta"
output_file = "aligned.fasta"

# Construct the command as a list
cmd = ["muscle", "-align", input_file, "-output", output_file]

# Execute the command
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode == 0:
    print("Alignment successful!")
else:
    print(f"Error: {result.stderr}")

Error: 
muscle 5.3.linux64 []  11.9Gb RAM, 12 cores
Built Jul 30 2025 21:13:04
(C) Copyright 2004-2021 Robert C. Edgar.
https://drive5.com

[align input.fasta]

---Fatal error---
Cannot open input.fasta, errno=2 No such file or directory

